In [1]:
import pandas as pd

df = pd.read_csv("../OPENACTIVE_MERGED.csv")
gap = pd.read_csv("../gapscore_merged.csv")

supply = df.groupby("borough")["session_count"].sum(min_count=1).reset_index(name="sessions")
venues = df.groupby("borough")["location_name"].nunique().reset_index(name="venues")
supply = supply.merge(venues, on="borough", how="outer")

demand = gap.groupby("borough").agg(
    inactive=("pct_inactive", "mean"),
    respondents=("respondents", "sum")
).reset_index()

g = demand.merge(supply, on="borough", how="outer")
g["sessions"] = g["sessions"].fillna(0)
g["venues"] = g["venues"].fillna(0)

full = g.copy()  # keep all 33 boroughs for reference
g = g[g["borough"] != "City of London"].copy()
# excluded: ~8k residents, no residential comparison base, and zero venues across every
# provider in the dataset - not comparable to a normal borough

print(g.shape)

(32, 5)


In [2]:
matrix = {
    ("high", "low"): "genuine desert",
    ("high", "mid"): "emerging desert",
    ("high", "high"): "under-monitored",
    ("mid", "low"): "blind spot risk",
    ("mid", "mid"): "average",
    ("mid", "high"): "well-served (moderate need)",
    ("low", "low"): "low need, low supply",
    ("low", "mid"): "adequately served",
    ("low", "high"): "well-served",
}

def classify(row, s_col, i_col):
    if row["sessions"] == 0:
        return "blind spot" if row[i_col] in ["mid", "high"] else "low need, low supply"
    return matrix.get((row[i_col], row[s_col]), "unclassified")

def bin_col(series, n, labels):
    return pd.qcut(series.rank(method="first"), n, labels=labels)

In [3]:
#main classiaction

g["s_bin"] = bin_col(g["sessions"], 3, ["low", "mid", "high"])
g["i_bin"] = bin_col(g["inactive"], 3, ["low", "mid", "high"])
g["class"] = g.apply(lambda r: classify(r, "s_bin", "i_bin"), axis=1)

print(g.sort_values("inactive", ascending=False)[["borough","inactive","sessions","class"]])

                   borough   inactive  sessions                        class
0     Barking and Dagenham  35.701377    2535.0              under-monitored
24                  Newham  30.470825    3196.0              under-monitored
3                    Brent  29.737033    2587.0              under-monitored
17                Hounslow  28.838464     646.0              emerging desert
16              Hillingdon  28.583208     231.0              emerging desert
25               Redbridge  28.517625       0.0                   blind spot
14                  Harrow  28.072601    1571.0              emerging desert
9                  Enfield  27.185417      77.0               genuine desert
15                Havering  27.001429    5696.0              under-monitored
8                   Ealing  25.890150    3427.0              under-monitored
2                   Bexley  25.494695     475.0              emerging desert
30          Waltham Forest  25.060002    5825.0  well-served (moderate need)

In [4]:
'''sensitivity check'''
concern = {
    "genuine desert": "high", "blind spot": "high",
    "emerging desert": "mid", "under-monitored": "mid", "blind spot risk": "mid",
    "average": "low", "well-served (moderate need)": "low", "adequately served": "low",
    "well-served": "low", "low need, low supply": "low",
}

g["s_bin2"] = bin_col(g["sessions"], 2, ["low", "high"])
g["i_bin2"] = bin_col(g["inactive"], 2, ["low", "high"])
median_matrix = {("high", "low"): "genuine desert", ("high", "high"): "under-monitored",
                 ("low", "low"): "low need, low supply", ("low", "high"): "well-served"}
g["class_median"] = g.apply(
    lambda r: "blind spot" if r["sessions"] == 0 and r["i_bin2"] == "high"
    else median_matrix.get((r["i_bin2"], r["s_bin2"]), "unclassified"), axis=1
)

In [5]:
g["s_bin4"] = bin_col(g["sessions"], 4, ["low", "mid_low", "mid_high", "high"])
g["i_bin4"] = bin_col(g["inactive"], 4, ["low", "mid_low", "mid_high", "high"])
quartile_matrix = {
    ("high", "low"): "genuine desert", ("high", "mid_low"): "emerging desert",
    ("high", "mid_high"): "under-monitored", ("high", "high"): "under-monitored",
    ("mid_high", "low"): "blind spot risk", ("mid_high", "mid_low"): "average",
    ("mid_high", "mid_high"): "average", ("mid_high", "high"): "well-served (moderate need)",
    ("mid_low", "low"): "blind spot risk", ("mid_low", "mid_low"): "average",
    ("mid_low", "mid_high"): "average", ("mid_low", "high"): "well-served (moderate need)",
    ("low", "low"): "low need, low supply", ("low", "mid_low"): "adequately served",
    ("low", "mid_high"): "adequately served", ("low", "high"): "well-served",
}
g["class_quartile"] = g.apply(
    lambda r: "blind spot" if r["sessions"] == 0 and r["i_bin4"] in ["mid_high", "high"]
    else quartile_matrix.get((r["i_bin4"], r["s_bin4"]), "unclassified"), axis=1
)

In [6]:
g["concern"] = g["class"].map(concern)
g["concern_median"] = g["class_median"].map(concern)
g["concern_quartile"] = g["class_quartile"].map(concern)
g["stable"] = (g["concern"] == g["concern_median"]) & (g["concern"] == g["concern_quartile"])

print("unstable boroughs (sensitive to threshold choice):")
print(g.loc[~g["stable"], ["borough", "concern", "concern_median", "concern_quartile"]])

unstable boroughs (sensitive to threshold choice):
           borough concern concern_median concern_quartile
2           Bexley     mid           high              low
4          Bromley     low            low              mid
7          Croydon     mid           high              mid
8           Ealing     mid            mid              low
10       Greenwich     mid           high              mid
15        Havering     mid            mid              low
16      Hillingdon     mid           high              mid
28          Sutton     low            mid              low
29   Tower Hamlets     low            mid              low
30  Waltham Forest     low            mid              low


In [7]:
london_avg = (g["inactive"] * g["respondents"]).sum() / g["respondents"].sum()
k = g["respondents"].median()
g["inactive_shrunk"] = ((g["inactive"] * g["respondents"]) + (london_avg * k)) / (g["respondents"] + k)

g["i_bin_shrunk"] = bin_col(g["inactive_shrunk"], 3, ["low", "mid", "high"])
changed = g[g["i_bin"] != g["i_bin_shrunk"]]
print("boroughs that moved tertile after shrinkage:")
print(changed[["borough", "inactive", "inactive_shrunk"]])

boroughs that moved tertile after shrinkage:
Empty DataFrame
Columns: [borough, inactive, inactive_shrunk]
Index: []


In [11]:
g.loc[g["borough"] == "Enfield", "class"] = "blind spot"
g.loc[g["borough"] == "Enfield", "note"] = (
    "reclassified from genuine desert - real centres exist (Albany, Southbury, Southgate, "
    "Edmonton), all run by Better, a feed with known geocoding/reporting issues"
)
g.loc[g["borough"] == "Redbridge", "note"] = (
    "confirmed blind spot - real centres exist (Redbridge Sports & Leisure, Wanstead, "
    "Fullwell Cross) but none appear in OpenActive under any provider"
)
g.loc[g["borough"] == "Barking and Dagenham", "note"] = (
    "genuine puzzle, not a data issue - real, recently upgraded Everyone Active centres, "
    "high supply confirmed, yet highest inactivity in London - demand-side barrier likely"
)

g.to_csv("../GAP_data/gap_scores.csv", index=False)
full.to_csv("../GAP_data/gap_scores_full33.csv", index=False)
print("saved GAP_data/gap_scores.csv and GAP_data/gap_scores_full33.csv")

saved GAP_data/gap_scores.csv and GAP_data/gap_scores_full33.csv
